# 03 · Sources, seeds & freshness

**Sources** declare the raw tables *other* tools load, so dbt can document
them, draw them in the DAG, and check their freshness.
**Seeds** are small static CSVs that dbt itself loads -- lookup tables owned
by analysts, not source data.

Companion reading: [docs/05](../docs/05_sources_seeds_freshness.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Simulate a fresh sync, then check freshness

`dbt source freshness` compares each source's `loaded_at_field`
(`_synced_at` here) against the `warn_after` / `error_after` rules in
`_staging__sources.yml`.

In [ ]:
with engine.begin() as c:
    c.execute(text("update raw.raw_orders set _synced_at = now()"))
    c.execute(text("update raw.raw_events set _synced_at = now()"))

res = run_dbt(["source", "freshness"])
assert res.success

## 2. The freshness artifact: `target/sources.json`

In [ ]:
fresh = json.loads(Path("target/sources.json").read_text())
pd.DataFrame(
    [{"source": r["unique_id"].split(".")[-2] + "." + r["unique_id"].split(".")[-1],
      "status": r["status"],
      "max_loaded_at": r.get("max_loaded_at", "")} for r in fresh["results"]]
)

Only tables with freshness rules are checked -- `raw_products` opted out
with `freshness: null` (a static catalog should not page anyone).

## 3. Now break it: age the orders sync by 3 days

`error_after` for orders is 72h, so this must FAIL the freshness check.

In [ ]:
with engine.begin() as c:
    c.execute(text("update raw.raw_orders set _synced_at = now() - interval '3 days'"))

res = run_dbt(["source", "freshness"])
assert not res.success, "expected freshness to fail!"

fresh = json.loads(Path("target/sources.json").read_text())
{r["unique_id"].split(".")[-1]: r["status"] for r in fresh["results"]}

In production this exact command runs on a schedule: a failing source
freshness check tells you the *pipeline upstream of dbt* is broken -- before
stakeholders find stale dashboards.

## 4. Repair the sync (cleanup)

In [ ]:
with engine.begin() as c:
    c.execute(text("update raw.raw_orders set _synced_at = now()"))

res = run_dbt(["source", "freshness"])
assert res.success

## 5. Seeds: version-controlled lookup CSVs

`payment_methods.csv` and `country_codes.csv` live in `seeds/`. They get
column type overrides, tests and docs like any model -- and models
`ref()` them exactly like models.

In [ ]:
!dbt seed --full-refresh

In [ ]:
%%sql
select * from dev_seeds.payment_methods order by payment_method

**Seed rules of thumb**: small (hundreds of rows), static, business-owned
mappings -> seed. Actual source data, big files, anything that changes on
its own -> NOT a seed, that's an EL tool's job.